In [11]:
!pip install -q transformers torch pypdf scikit-learn sentencepiece

In [12]:
!pip install PyPDF2 -q
!pip install sentence-transformers -q
!pip install pymupdf -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 232.6/232.6 kB 6.0 MB/s eta 0:00:00a 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.9/24.9 MB 75.0 MB/s eta 0:00:00:00:0100:01


In [13]:
# !mkdir -p src

In [14]:
%%writefile src/config.py
MODEL_NAME = "sentence-transformers/all-mpnet-base-v2"

CHUNK_SIZE = 180
CHUNK_OVERLAP = 50
TOP_K = 3

DATA_FOLDER = "/kaggle/input/datasets/sakshamshah/documents-source"

MIN_CHARS_PER_PAGE = 40

Overwriting src/config.py


In [15]:
!ls /kaggle/input

datasets


In [16]:
%%writefile src/loader.py
import os
import re
from collections import Counter
import fitz  # PyMuPDF

from src.config import MIN_CHARS_PER_PAGE


def normalize_text(text):
    text = text.replace("\xa0", " ")
    text = re.sub(r"[ \t]+", " ", text)
    text = re.sub(r"\n\s*\n+", "\n\n", text)
    return text.strip()


def clean_lines(lines):
    cleaned = []

    for line in lines:
        line = line.strip()

        if not line:
            continue

        if re.fullmatch(r"[-–—]?\s*\d+\s*[-–—]?", line):
            continue

        alpha_count = sum(ch.isalpha() for ch in line)
        if alpha_count == 0 and len(line) < 8:
            continue

        cleaned.append(line)

    return cleaned


def remove_repeated_lines_across_pages(all_pages_lines):
    if len(all_pages_lines) < 3:
        return all_pages_lines

    counter = Counter()

    for lines in all_pages_lines:
        counter.update(set(lines))

    repeated_lines = {
        line for line, count in counter.items()
        if count >= max(3, int(0.5 * len(all_pages_lines))) and len(line.split()) <= 12
    }

    cleaned_pages = []
    for lines in all_pages_lines:
        cleaned_pages.append([line for line in lines if line not in repeated_lines])

    return cleaned_pages


def detect_text_mode(lines, page_text):
    if len(page_text.strip()) < MIN_CHARS_PER_PAGE:
        return "low_text"

    word_counts = [len(line.split()) for line in lines if line.strip()]
    if not word_counts:
        return "low_text"

    avg_words_per_line = sum(word_counts) / len(word_counts)
    short_line_ratio = sum(1 for wc in word_counts if wc <= 6) / len(word_counts)

    punctuation_count = len(re.findall(r"[.!?]", page_text))
    total_words = max(1, len(page_text.split()))
    punctuation_ratio = punctuation_count / total_words

    if avg_words_per_line >= 7 and short_line_ratio < 0.45 and punctuation_ratio >= 0.02:
        return "paragraph"

    return "line"


def is_junk_page(page_text, lines):
    text_lower = page_text.lower()

    junk_terms = [
        "table of contents",
        "contents",
        "list of figures",
        "list of tables",
        "statutory declaration",
        "declaration",
        "table of abbreviations",
        "abbreviations",
    ]

    for term in junk_terms:
        if term in text_lower:
            return True

    dot_leader_lines = sum(1 for line in lines if re.search(r"\.{2,}\s*\d+\s*$", line))
    if dot_leader_lines >= 3:
        return True

    if lines:
        short_lines = sum(1 for line in lines if len(line.split()) <= 6)
        if short_lines / len(lines) > 0.8 and dot_leader_lines >= 1:
            return True

    return False


def load_txt_file(filepath):
    with open(filepath, "r", encoding="utf-8", errors="ignore") as f:
        text = f.read()

    text = normalize_text(text)

    return [{
        "source": os.path.basename(filepath),
        "file_type": "txt",
        "page": None,
        "text_mode": "paragraph",
        "text": text
    }]


def extract_page_text_with_pymupdf(page):
    blocks = page.get_text("blocks")
    blocks = sorted(blocks, key=lambda b: (round(b[1], 1), round(b[0], 1)))

    lines = []
    for block in blocks:
        block_text = block[4]
        if not block_text:
            continue

        for line in block_text.split("\n"):
            line = line.strip()
            if line:
                lines.append(line)

    return lines


def load_pdf_file(filepath):
    pages_raw = []

    doc = fitz.open(filepath)

    for page_num in range(len(doc)):
        page = doc[page_num]

        try:
            lines = extract_page_text_with_pymupdf(page)
        except Exception:
            lines = []

        lines = clean_lines(lines)

        pages_raw.append({
            "page": page_num + 1,
            "lines": lines
        })

    doc.close()

    all_pages_lines = [item["lines"] for item in pages_raw]
    all_pages_lines = remove_repeated_lines_across_pages(all_pages_lines)

    documents = []

    for i, item in enumerate(pages_raw):
        lines = all_pages_lines[i]
        page_text = "\n".join(lines)
        page_text = normalize_text(page_text)

        if is_junk_page(page_text, lines):
            continue

        text_mode = detect_text_mode(lines, page_text)

        if text_mode == "low_text":
            continue

        documents.append({
            "source": os.path.basename(filepath),
            "file_type": "pdf",
            "page": item["page"],
            "text_mode": text_mode,
            "text": page_text
        })

    return documents


def load_documents(folder_path):
    documents = []

    for filename in sorted(os.listdir(folder_path)):
        filepath = os.path.join(folder_path, filename)

        if not os.path.isfile(filepath):
            continue

        ext = os.path.splitext(filename)[1].lower()

        try:
            if ext == ".txt":
                documents.extend(load_txt_file(filepath))
            elif ext == ".pdf":
                documents.extend(load_pdf_file(filepath))
        except Exception as e:
            print(f"Skipping {filename} because of error: {e}")

    return documents

Writing src/loader.py


In [17]:
%%writefile src/chunker.py
import re
from src.config import CHUNK_SIZE, CHUNK_OVERLAP


def split_into_sentences(text):
    sentences = re.split(r'(?<=[.!?])\s+', text.strip())
    return [s.strip() for s in sentences if s.strip()]


def is_junk_chunk(text):
    text_lower = text.lower().strip()
    lines = [line.strip() for line in text.split("\n") if line.strip()]

    if not text_lower:
        return True

    # obvious front-matter/navigation keywords
    junk_terms = [
        "table of contents",
        "contents",
        "list of figures",
        "list of tables",
        "statutory declaration",
        "table of abbreviations",
        "abbreviations"
    ]
    for term in junk_terms:
        if term in text_lower:
            return True

    # dot-leader contents pattern
    dot_leader_lines = sum(1 for line in lines if re.search(r"\.{2,}\s*\d+\s*$", line))
    if dot_leader_lines >= 2:
        return True

    # too many short numbered navigation lines like:
    # 4.2 Tourism case study .... 63
    numbered_nav_lines = 0
    for line in lines:
        if re.match(r"^\d+(\.\d+)*\s+", line) and re.search(r"\d+\s*$", line):
            numbered_nav_lines += 1

    if lines and numbered_nav_lines / len(lines) > 0.5:
        return True

    # almost everything is short lines and no real sentence structure
    short_lines = sum(1 for line in lines if len(line.split()) <= 6)
    sentence_like = len(re.findall(r"[.!?]", text))

    if lines and short_lines / len(lines) > 0.8 and sentence_like == 0:
        return True

    return False


def chunk_paragraph_text(text, chunk_size, overlap):
    sentences = split_into_sentences(text)
    chunks = []

    current_chunk = []
    current_length = 0

    for sentence in sentences:
        sentence_length = len(sentence.split())

        if current_length + sentence_length <= chunk_size:
            current_chunk.append(sentence)
            current_length += sentence_length
        else:
            if current_chunk:
                chunk_text = " ".join(current_chunk).strip()
                chunks.append(chunk_text)

                overlap_words = chunk_text.split()[-overlap:] if overlap > 0 else []
                overlap_text = " ".join(overlap_words)

                current_chunk = [overlap_text, sentence] if overlap_text else [sentence]
                current_length = len(" ".join(current_chunk).split())
            else:
                chunks.append(sentence)

    if current_chunk:
        chunks.append(" ".join(current_chunk).strip())

    return chunks


def chunk_line_text(text, chunk_size, overlap):
    lines = [line.strip() for line in text.split("\n") if line.strip()]
    chunks = []

    current_chunk = []
    current_length = 0

    for line in lines:
        line_length = len(line.split())

        if current_length + line_length <= chunk_size:
            current_chunk.append(line)
            current_length += line_length
        else:
            if current_chunk:
                chunk_text = "\n".join(current_chunk).strip()
                chunks.append(chunk_text)

                overlap_words = chunk_text.split()[-overlap:] if overlap > 0 else []
                overlap_text = " ".join(overlap_words)

                current_chunk = [overlap_text, line] if overlap_text else [line]
                current_length = len(" ".join(current_chunk).split())
            else:
                chunks.append(line)

    if current_chunk:
        chunks.append("\n".join(current_chunk).strip())

    return chunks


def build_chunk_index(documents, chunk_size=CHUNK_SIZE, overlap=CHUNK_OVERLAP):
    chunk_index = []

    line_chunk_size = max(40, chunk_size // 2)
    line_overlap = max(10, overlap // 2)

    for doc in documents:
        text = doc["text"].strip()
        if not text:
            continue

        text_mode = doc.get("text_mode", "paragraph")

        if text_mode == "paragraph" or doc["file_type"] == "txt":
            chunks = chunk_paragraph_text(text, chunk_size, overlap)

        elif text_mode == "line":
            chunks = chunk_line_text(text, line_chunk_size, line_overlap)

        else:
            continue

        for i, chunk in enumerate(chunks):
            chunk = chunk.strip()

            if not chunk:
                continue

            if is_junk_chunk(chunk):
                continue

            chunk_index.append({
                "source": doc["source"],
                "file_type": doc["file_type"],
                "page": doc["page"],
                "text_mode": text_mode,
                "chunk_id": i,
                "text": chunk
            })

    return chunk_index

Writing src/chunker.py


In [18]:
%%writefile src/embedder.py
import torch
from transformers import AutoTokenizer, AutoModel


class TextEmbedder:
    def __init__(self, model_name: str):
        self.tokenizer = AutoTokenizer.from_pretrained(model_name)
        self.model = AutoModel.from_pretrained(model_name)

    def mean_pooling(self, model_output, attention_mask):
        token_embeddings = model_output[0]
        input_mask_expanded = attention_mask.unsqueeze(-1).expand(token_embeddings.size()).float()

        return torch.sum(token_embeddings * input_mask_expanded, dim=1) / torch.clamp(
            input_mask_expanded.sum(dim=1), min=1e-9
        )

    def encode(self, texts):
        encoded_input = self.tokenizer(
            texts,
            padding=True,
            truncation=True,
            return_tensors="pt"
        )

        with torch.no_grad():
            model_output = self.model(**encoded_input)

        embeddings = self.mean_pooling(model_output, encoded_input["attention_mask"])
        embeddings = torch.nn.functional.normalize(embeddings, p=2, dim=1)

        return embeddings.cpu().numpy()

Writing src/embedder.py


In [19]:
%%writefile src/generator.py
import re
from sklearn.metrics.pairwise import cosine_similarity


class AnswerGenerator:
    def split_into_sentences(self, text):
        sentences = re.split(r'(?<=[.!?])\s+', text.strip())
        return [s.strip() for s in sentences if s.strip()]

    def clean_sentences(self, sentences):
        cleaned = []

        for s in sentences:
            if len(s.split()) < 6:
                continue

            if re.search(r"\.{2,}\s*\d+$", s):
                continue

            if re.fullmatch(r"[\d\.]+\s+.*\s+\d+", s):
                continue

            cleaned.append(s)

        return cleaned

    def generate_answer(self, query, results, embedder, max_sentences=2):
        if not results:
            return "The answer is not available in the retrieved documents."

        context = " ".join(item["text"] for item in results)
        sentences = self.split_into_sentences(context)
        sentences = self.clean_sentences(sentences)

        if not sentences:
            return "The answer is not available in the retrieved documents."

        sentence_embeddings = embedder.encode(sentences)
        query_embedding = embedder.encode([query])

        scores = cosine_similarity(query_embedding, sentence_embeddings)[0]
        ranked_indices = scores.argsort()[::-1]

        selected = []
        seen = set()

        for idx in ranked_indices:
            sentence = sentences[idx]

            if sentence not in seen:
                selected.append(sentence)
                seen.add(sentence)

            if len(selected) == max_sentences:
                break

        if not selected:
            return "The answer is not available in the retrieved documents."

        return " ".join(selected)

Writing src/generator.py


In [20]:
%%writefile src/search.py
import re
import numpy as np
from sklearn.metrics.pairwise import cosine_similarity


def keyword_overlap_score(query, text):
    query_words = set(re.findall(r"\b\w+\b", query.lower()))
    text_words = set(re.findall(r"\b\w+\b", text.lower()))

    if not query_words:
        return 0.0

    overlap = query_words.intersection(text_words)
    return len(overlap) / len(query_words)


def junk_penalty(text):
    text_lower = text.lower()
    penalty = 0.0

    junk_terms = [
        "contents",
        "table of contents",
        "list of figures",
        "list of tables",
        "references"
    ]

    for term in junk_terms:
        if term in text_lower:
            penalty += 0.15

    if re.search(r"\.{2,}\s*\d+", text_lower):
        penalty += 0.15

    return penalty


def semantic_search(query, chunk_index, chunk_embeddings, embedder, top_k=3, source_filter=None):
    filtered_chunks = []
    filtered_embeddings = []

    for i, chunk in enumerate(chunk_index):
        if source_filter is None or chunk["source"] == source_filter:
            filtered_chunks.append(chunk)
            filtered_embeddings.append(chunk_embeddings[i])

    if not filtered_chunks:
        return []

    filtered_embeddings = np.array(filtered_embeddings)

    query_embedding = embedder.encode([query])
    semantic_scores = cosine_similarity(query_embedding, filtered_embeddings)[0]

    final_scores = []

    for i, chunk in enumerate(filtered_chunks):
        lexical_score = keyword_overlap_score(query, chunk["text"])
        penalty = junk_penalty(chunk["text"])

        score = (0.85 * semantic_scores[i]) + (0.20 * lexical_score) - penalty
        final_scores.append(score)

    final_scores = np.array(final_scores)
    top_indices = np.argsort(final_scores)[::-1][:top_k]

    results = []

    for idx in top_indices:
        results.append({
            "score": float(final_scores[idx]),
            "source": filtered_chunks[idx]["source"],
            "page": filtered_chunks[idx]["page"],
            "chunk_id": filtered_chunks[idx]["chunk_id"],
            "text_mode": filtered_chunks[idx]["text_mode"],
            "text": filtered_chunks[idx]["text"]
        })

    return results

Writing src/search.py


In [21]:
%%writefile src/reranker.py
from sentence_transformers import CrossEncoder


class ChunkReranker:
    def __init__(self, model_name="cross-encoder/ms-marco-MiniLM-L-6-v2"):
        self.model = CrossEncoder(model_name)

    def rerank(self, query, results, top_k=None):
        if not results:
            return []

        pairs = [(query, item["text"]) for item in results]
        scores = self.model.predict(pairs)

        reranked = []
        for item, score in zip(results, scores):
            new_item = item.copy()
            new_item["rerank_score"] = float(score)
            reranked.append(new_item)

        reranked.sort(key=lambda x: x["rerank_score"], reverse=True)

        if top_k is not None:
            reranked = reranked[:top_k]

        return reranked

Writing src/reranker.py


In [22]:
from importlib import reload
import src.config
import src.loader
import src.chunker
import src.search
import src.generator
import src.embedder
import src.reranker

reload(src.config)
reload(src.loader)
reload(src.chunker)
reload(src.search)
reload(src.generator)
reload(src.embedder)
reload(src.reranker)

from src.config import MODEL_NAME, CHUNK_SIZE, CHUNK_OVERLAP, TOP_K, DATA_FOLDER
from src.loader import load_documents
from src.chunker import build_chunk_index
from src.embedder import TextEmbedder
from src.search import semantic_search
from src.generator import AnswerGenerator
from src.reranker import ChunkReranker

In [23]:
documents = load_documents(DATA_FOLDER)
print("Loaded items:", len(documents))

Loaded items: 156


In [24]:
embedder = TextEmbedder(MODEL_NAME)
generator = AnswerGenerator()
reranker = ChunkReranker()

config.json:   0%|          | 0.00/571 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/363 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

...

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/132 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

In [25]:
DEV_FILES = [
    "Master_thesis_saksham_shah.pdf",
    "03_E3.1_AAL2_InverseDynamics_SS2024.pdf"
]

In [26]:
selected_documents = [doc for doc in documents if doc["source"] in DEV_FILES]
print("Selected development items:", len(selected_documents))

dev_chunk_index = build_chunk_index(
    documents=selected_documents,
    chunk_size=CHUNK_SIZE,
    overlap=CHUNK_OVERLAP
)

dev_chunk_texts = [item["text"] for item in dev_chunk_index]
dev_chunk_embeddings = embedder.encode(dev_chunk_texts)

print("Development chunks:", len(dev_chunk_index))
print("Development embeddings shape:", dev_chunk_embeddings.shape)

Selected development items: 156
Development chunks: 371
Development embeddings shape: (371, 768)


In [27]:
VAL_FOLDER = "/kaggle/input/datasets/sakshamshah/resume-ethics"

In [28]:
val_documents = load_documents(VAL_FOLDER)
print("Selected validation items:", len(val_documents))

Selected validation items: 17


In [29]:
val_chunk_index = build_chunk_index(
    documents=val_documents,
    chunk_size=CHUNK_SIZE,
    overlap=CHUNK_OVERLAP
)

val_chunk_texts = [item["text"] for item in val_chunk_index]

if not val_chunk_texts:
    print("Validation documents loaded, but no chunks were created.")
else:
    val_chunk_embeddings = embedder.encode(val_chunk_texts)
    print("Validation chunks:", len(val_chunk_index))
    print("Validation embeddings shape:", val_chunk_embeddings.shape)

Validation chunks: 97
Validation embeddings shape: (97, 768)


In [36]:
query = "How does the thesis model holiday effects differently from standard dummy variables?"

results = semantic_search(
    query=query,
    chunk_index=dev_chunk_index,
    chunk_embeddings=dev_chunk_embeddings,
    embedder=embedder,
    top_k=TOP_K,
    source_filter="Master_thesis_saksham_shah.pdf"
)

answer = generator.generate_answer(query, results, embedder)

print("Question:", query)
print("\nAnswer:\n")
print(answer)
print("\nSources used:")
for item in results:
    print("-", item["source"], "| page", item["page"], "| chunk", item["chunk_id"])

Question: How does the thesis model holiday effects differently from standard dummy variables?

Answer:

This reflects
the modelling assumption that nearby holidays may jointly influence activity and is
consistent with standard additive regression formulations for calendar effects. long-run level is absorbed by the intercept rather than by the holiday regressor, a standard identifiability device for dummy variables in time series regression [11].

Sources used:
- Master_thesis_saksham_shah.pdf | page 33 | chunk 1
- Master_thesis_saksham_shah.pdf | page 97 | chunk 0
- Master_thesis_saksham_shah.pdf | page 36 | chunk 3


In [37]:
query = "What problem does the proposed method try to solve?"

results = semantic_search(
    query=query,
    chunk_index=dev_chunk_index,
    chunk_embeddings=dev_chunk_embeddings,
    embedder=embedder,
    top_k=TOP_K,
    source_filter="Master_thesis_saksham_shah.pdf"
)

answer = generator.generate_answer(query, results, embedder)

print("Question:", query)
print("\nAnswer:\n")
print(answer)
print("\nSources used:")
for item in results:
    print("-", item["source"], "| page", item["page"], "| chunk", item["chunk_id"])

Question: What problem does the proposed method try to solve?

Answer:

While Appendix A
Supplementary Implementation Figures
and Diagnostics
This appendix collects supporting figures and minimal code excerpts that illustrate key
steps of the experimental pipeline described in Chapter 3. The same scheme is applied on validation for model selection, after which the chosen
configuration is refit on train+validation and applied once to the held out test span.

Sources used:
- Master_thesis_saksham_shah.pdf | page 43 | chunk 0
- Master_thesis_saksham_shah.pdf | page 44 | chunk 2
- Master_thesis_saksham_shah.pdf | page 107 | chunk 0


In [35]:
query = "What are the main steps of inverse dynamics?"

results = semantic_search(
    query=query,
    chunk_index=dev_chunk_index,
    chunk_embeddings=dev_chunk_embeddings,
    embedder=embedder,
    top_k=TOP_K,
    source_filter="03_E3.1_AAL2_InverseDynamics_SS2024.pdf"
)

answer = generator.generate_answer(query, results, embedder)

print("Question:", query)
print("\nAnswer:\n")
print(answer)
print("\nSources used:")
for item in results:
    print("-", item["source"], "| page", item["page"], "| chunk", item["chunk_id"])

Question: What are the main steps of inverse dynamics?

Answer:

slide 9
Inverse Dynamics – Determination of muscle forces & moments
What do the different muscles do? INVERSE DYNAMICS – DYNAMIC EXAMPLE
Active and Assisted Living 2
slide 44
Estimation of muscles forces and joint moments INVERSE DYNAMICS – STATIC EXAMPLE
Active and Assisted Living 2
slide 29
Estimation of muscles forces and joint moments

Sources used:
- 03_E3.1_AAL2_InverseDynamics_SS2024.pdf | page 9 | chunk 0
- 03_E3.1_AAL2_InverseDynamics_SS2024.pdf | page 43 | chunk 0
- 03_E3.1_AAL2_InverseDynamics_SS2024.pdf | page 28 | chunk 0


In [39]:
query = "What is calculated in inverse dynamics?"

results = semantic_search(
    query=query,
    chunk_index=dev_chunk_index,
    chunk_embeddings=dev_chunk_embeddings,
    embedder=embedder,
    top_k=TOP_K,
    source_filter="03_E3.1_AAL2_InverseDynamics_SS2024.pdf"
)

answer = generator.generate_answer(query, results, embedder)

print("Question:", query)
print("\nAnswer:\n")
print(answer)
print("\nSources used:")
for item in results:
    print("-", item["source"], "| page", item["page"], "| chunk", item["chunk_id"])

Question: What is calculated in inverse dynamics?

Answer:

INVERSE DYNAMICS – DYNAMIC EXAMPLE
Active and Assisted Living 2
slide 44
Estimation of muscles forces and joint moments INVERSE DYNAMICS – STATIC EXAMPLE
Active and Assisted Living 2
slide 29
Estimation of muscles forces and joint moments slide 9
Inverse Dynamics – Determination of muscle forces & moments
What do the different muscles do?

Sources used:
- 03_E3.1_AAL2_InverseDynamics_SS2024.pdf | page 9 | chunk 0
- 03_E3.1_AAL2_InverseDynamics_SS2024.pdf | page 43 | chunk 0
- 03_E3.1_AAL2_InverseDynamics_SS2024.pdf | page 28 | chunk 0


In [34]:
query = "How does the document explain fairness and bias in data science?"

results = semantic_search(
    query=query,
    chunk_index=val_chunk_index,
    chunk_embeddings=val_chunk_embeddings,
    embedder=embedder,
    top_k=TOP_K,
    source_filter="Data Science Ethics Introduction (1).pdf"
)

answer = generator.generate_answer(query, results, embedder)


print("Question:", query)
print("\nValidation Answer:\n")
print(answer)
print("\nSources used:")
for item in results:
    print("-", item["source"], "| page", item["page"], "| chunk", item["chunk_id"])

Question: How does the document explain fairness and bias in data science?

Validation Answer:

This leads to an equilibrium of data science somewhat similar to that of
the top of Figure 1.2, with practices that typically do not use gender in the pre-
diction model, include inherently comprehensible models and have stringent
evaluate such explanations are not yet perfectly defined, and avoiding discrim-
privacy policies. Next to this fairness issue at the feature dimension (on discrimination), fair-
ness also relates to the instance dimension (on privacy).

Sources used:
- Data Science Ethics Introduction (1).pdf | page 10 | chunk 0
- Data Science Ethics Introduction (1).pdf | page 16 | chunk 1
- Data Science Ethics Introduction (1).pdf | page 11 | chunk 4


In [40]:
query = "How does the document describe the social impact of data science??"

results = semantic_search(
    query=query,
    chunk_index=val_chunk_index,
    chunk_embeddings=val_chunk_embeddings,
    embedder=embedder,
    top_k=TOP_K,
    source_filter="Data Science Ethics Introduction (1).pdf"
)

answer = generator.generate_answer(query, results, embedder)


print("Question:", query)
print("\nValidation Answer:\n")
print(answer)
print("\nSources used:")
for item in results:
    print("-", item["source"], "| page", item["page"], "| chunk", item["chunk_id"])

Question: How does the document describe the social impact of data science??

Validation Answer:

However, just as with any technology, data science has also come with some negative consequences: an increase of privacy invasion, data-driven discrimination against sensitive groups' and data-driven decision making without exPlanations. the importance and potential impact of data science ethics'
You might wonder, why is this important, and why should I care about data
science ethics?

Sources used:
- Data Science Ethics Introduction (1).pdf | page 5 | chunk 1
- Data Science Ethics Introduction (1).pdf | page 9 | chunk 0
- Data Science Ethics Introduction (1).pdf | page 5 | chunk 6


In [41]:
 source_name = input("Enter exact file name or press Enter for all development documents: ").strip()
 if source_name == "":
     source_name = None

 query = input("Enter your question: ").strip()

 results = semantic_search(
     query=query,
     chunk_index=dev_chunk_index,
     chunk_embeddings=dev_chunk_embeddings,
     embedder=embedder,     top_k=10,
     source_filter=source_name
 )

 reranked_results = reranker.rerank(query, results, top_k=3)

 answer = generator.generate_answer(query, reranked_results, embedder)

 print()
 print("Answer:")
 print(answer)
 print()

 if reranked_results:
     print("Sources used:")
     seen = set()
     for item in reranked_results:
         key = (item["source"], item["page"], item["chunk_id"])
         if key not in seen:
             print("-", item["source"], "| page", item["page"], "| chunk", item["chunk_id"])
             seen.add(key)
else:
     print("No matching results found.")